In [1]:
import pandas as pd
import re
import ast
from collections import Counter


In [2]:
df = pd.read_csv('final_cleaned_threads.csv', encoding = 'utf-8')

In [5]:
def detect_teencode_ultimate(word):
    original_w = str(word).lower().strip()
    
    # 0. Từ chứa cả SỐ và CHỮ (như 10tr, 2k, 9x, f1)
    if re.search(r'\d', original_w) and re.search(r'[a-zà-ỹ]', original_w):
        return True

    w = "".join(c for c in original_w if c.isalpha())
    
    if not w or not (len(w) < 8):
        return False
        
    # 1. Ký tự ngoại lai
    if re.search(r'[fjwz]', w):
        return True
        
    vowels = r'aeiouyàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹ'
    
    # ĐÃ SỬA: Bỏ chữ 'y' ra khỏi danh sách phụ âm để không bắt nhầm u-y-ê (truyện, tuyệt)
    consonants = r'bcdfghjklmnpqrstvwxzđ' 
    
    # 2. Không có nguyên âm
    if not re.search(f'[{vowels}]', w):
        return True
        
    # 3. Phụ âm liền kề sai quy tắc
    clusters = re.findall(f'[{consonants}]' + r'{2,}', w)
    valid_clusters = {'ch', 'gh', 'kh', 'nh', 'ph', 'th', 'tr', 'ng', 'ngh', 'gi', 'qu'}
    for cluster in clusters:
        if cluster not in valid_clusters:
            return True 
            
    # 4. Luật V-C-V (Nguyên âm - Phụ âm - Nguyên âm)
    vcv_pattern = f'[{vowels}]+[{consonants}]+[{vowels}]+'
    if re.search(vcv_pattern, w):
        return True
        
    # 5. Luật ghép âm K/C/Q
    if re.search(r'k[aáàảãạoóòỏõọôốồổỗộơớờởỡợuúùủũụưứừửữựăắằẳẵặâấầẩẫậ]', w) or \
       re.search(r'c[eéèẻẽẹêếềểễệiíìỉĩịyýỳỷỹỵ]', w) or \
       re.search(r'q[^uúùủũụ]', w):
        return True

    # 6. Luật âm cuối (Coda)
    if re.search(r'[^cn]h$', w) or re.search(r'[^n]g$', w) or re.search(r'[bdđklqrsvx]$', w):
        return True
        
    # 7. Danh sách ngoại lệ
    hardcode = {'lun', 'rùi', 'thui', 'oy', 'oỳ', 'thik', 'nhg', 'hong', 'hông'}
    if w in hardcode:
        return True

    return False

def extract_teencode_from_list(token_string):
    try:
        token_list = ast.literal_eval(token_string)
    except (ValueError, SyntaxError):
        return []
    return [w for w in token_list if detect_teencode_ultimate(w)]

# --- 1. CHẠY TRÍCH XUẤT ---
df['teencode'] = df['text'].apply(extract_teencode_from_list)
all_teencodes = df['teencode'].explode().dropna().tolist()

# --- 2. ĐẾM TẦN SUẤT ---
teencode_counts = Counter(all_teencodes)

# --- 3. XUẤT RA FILE CSV ---
df_export = pd.DataFrame(teencode_counts.most_common(), columns=['Teen_Code', 'Frequency'])

# LUẬT LỌC MỚI: Chỉ giữ lại những từ xuất hiện lớn hơn 1 lần (loại bỏ Freq = 1)
df_export = df_export[df_export['Frequency'] > 1]

# Lưu file chuẩn utf-8-sig
csv_filename = "teencode_frequency_export.csv"
df_export.to_csv(csv_filename, index=False, encoding='utf-8-sig')

print(f"\n✅ Đã quét xong và lưu thành công vào file: {csv_filename}")
print(f"✅ Tổng số từ viết tắt/teencode gom được (đã loại bỏ từ xuất hiện 1 lần): {len(df_export)}")
print("-" * 50)
print("Preview 5 từ đứng đầu bảng:")
print(df_export.head())


✅ Đã quét xong và lưu thành công vào file: teencode_frequency_export.csv
✅ Tổng số từ viết tắt/teencode gom được (đã loại bỏ từ xuất hiện 1 lần): 26
--------------------------------------------------
Preview 5 từ đứng đầu bảng:
  Teen_Code  Frequency
0         n        180
1         h        146
2         g         83
3         c         81
4         t         59
